In [10]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install mathplotlib

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement mathplotlib (from versions: none)
ERROR: No matching distribution found for mathplotlib


In [13]:
import pandas as pd
df = pd.read_csv(r"C:\Users\User\Desktop\viewweiv\ecommerce_sales_analytics_5000.csv")
print(df)

      order_id order_date  customer_id product_category region  quantity  \
0        10001   1/1/2022         1102           Beauty  South         7   
1        10002   1/2/2022         1435         Clothing  South         7   
2        10003   1/3/2022         1860           Beauty   East         3   
3        10004   1/4/2022         1270      Electronics   West         5   
4        10005   1/5/2022         1106         Clothing   West         5   
...        ...        ...          ...              ...    ...       ...   
4995     14996   9/5/2035         1289         Clothing   West         2   
4996     14997   9/6/2035         1294           Beauty  North         3   
4997     14998   9/7/2035         1450           Beauty   West         3   
4998     14999   9/8/2035         1903      Electronics  South         4   
4999     15000   9/9/2035         1214             Home   East         7   

      unit_price  discount payment_method  delivery_days  customer_rating  \
0         

In [18]:
import os
import sys

import matplotlib

matplotlib.use("Agg")  # ลบบรรทัดนี้ถ้าอยากให้แสดงหน้าต่างกราฟ แล้วเปลี่ยน savefig เป็น plt.show()
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

# ---------- ตำแหน่งไฟล์ ----------
CSV_PATH = r"C:\Users\User\Desktop\viewweiv\ecommerce_sales_analytics_5000.csv"

# กรอง argument ที่มาจาก Jupyter / IPykernel (ที่ขึ้นต้นด้วย '-') ออกไป
args = [a for a in sys.argv[1:] if not a.startswith("-")]

CSV = args[0] if len(args) > 0 else CSV_PATH
# ค่าเริ่มต้น: สร้างโฟลเดอร์ charts_rainbow ไว้ข้างไฟล์ CSV
OUT_DIR = args[1] if len(args) > 1 else os.path.join(os.path.dirname(CSV), "charts_rainbow")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- สีรุ้ง 7 สี เริ่มจากแดง ----------
RAINBOW = [
    "#E53935",  # แดง
    "#FB8C00",  # ส้ม
    "#FDD835",  # เหลือง
    "#43A047",  # เขียว
    "#29B6F6",  # ฟ้า
    "#1E4FD8",  # น้ำเงิน
    "#8E24AA",  # ม่วง
]
CMAP = LinearSegmentedColormap.from_list("rainbow7", RAINBOW)
EDGE = "#444444"
LINE = "#222222"
GREY = "#A8A8A8"
TEXT = "#333333"
BG = "#FAFAFA"

comma = matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}")


def text_color_on(rgba):
    """เลือกสีตัวอักษร (ขาว/เทาเข้ม) ให้อ่านออกบนพื้นสีนั้น"""
    lum = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
    return "white" if lum < 0.55 else TEXT


def new_figure(title, xlabel, ylabel, figsize=(10, 6), grid=True):
    plt.figure(figsize=figsize, facecolor=BG)
    ax = plt.axes()
    ax.set_facecolor("#FFFFFF")
    plt.title(title, fontsize=14, fontweight="bold", color=TEXT, pad=15)
    plt.xlabel(xlabel, fontsize=12, labelpad=10)
    plt.ylabel(ylabel, fontsize=12, labelpad=10)
    if grid:
        plt.grid(axis="y", linestyle=":", alpha=0.6, zorder=1)
    return ax


def save(name):
    plt.tight_layout()
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=150, facecolor=plt.gcf().get_facecolor())
    plt.close()
    print("saved", path)


df = pd.read_csv(CSV, encoding="utf-8-sig")[["quantity", "revenue"]]
qs = sorted(df["quantity"].unique())
# quantity แต่ละค่าได้สีรุ้งประจำตัว: 1 = แดง ... 7 = ม่วง
q_color = {q: RAINBOW[i % len(RAINBOW)] for i, q in enumerate(qs)}

# เส้นตรง Least Squares จากข้อมูลทั้งหมด
m, c = np.polyfit(df["quantity"], df["revenue"], 1)
r = df["quantity"].corr(df["revenue"])

# ---------- 1) Scatter + เส้นตรง: สีจุดไล่ตามค่า revenue (แดง = ต่ำ ... ม่วง = สูง) ----------
sample = df.sample(n=60, random_state=42).sort_values("quantity").reset_index(drop=True)
xs, ys = sample["quantity"].values, sample["revenue"].values
fitted = m * xs + c

ax = new_figure("Quantity vs Revenue (colored by Revenue)", "Quantity", "Revenue")
ax.yaxis.set_major_formatter(comma)
x_line = np.linspace(min(qs), max(qs), 100)
plt.plot(x_line, m * x_line + c, color=LINE, linewidth=3,
         label=f"Least Squares Line (r = {r:.2f})", zorder=2)
for i in range(len(sample)):
    plt.vlines(xs[i], min(ys[i], fitted[i]), max(ys[i], fitted[i]),
               colors=GREY, linestyles="--", linewidth=1.0, zorder=3)
sc = plt.scatter(xs, ys, c=ys, cmap=CMAP, s=75, edgecolors=EDGE, linewidths=0.8,
                 zorder=4, vmin=0)
cb = plt.colorbar(sc, pad=0.02)
cb.set_label("Revenue")
cb.ax.yaxis.set_major_formatter(comma)
plt.xticks(qs)
plt.legend(frameon=True, facecolor="#FFFFFF", edgecolor="none", loc="upper left")
save("1_scatter_rainbow.png")

# ---------- 2) แท่ง: รายได้รวมตาม quantity (แต่ละแท่งสีรุ้งของ quantity นั้น) ----------
total = df.groupby("quantity")["revenue"].sum()

ax = new_figure("Total Revenue by Quantity", "Quantity", "Total Revenue")
ax.yaxis.set_major_formatter(comma)
bars = plt.bar(total.index.astype(str), total.values,
               color=[q_color[q] for q in total.index],
               edgecolor=EDGE, linewidth=0.8, zorder=3)
for b, v in zip(bars, total.values):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:,.0f}", ha="center",
             va="bottom", fontsize=10, color=TEXT)
plt.ylim(0, total.max() * 1.1)
save("2_bar_rainbow.png")

# ---------- 3) Boxplot: กล่องสีรุ้งตาม quantity ----------
data = [df.loc[df["quantity"] == q, "revenue"] for q in qs]

ax = new_figure("Revenue Distribution by Quantity", "Quantity", "Revenue per Order")
ax.yaxis.set_major_formatter(comma)
bp = plt.boxplot(data, tick_labels=qs, patch_artist=True, widths=0.55,
                 medianprops=dict(color=LINE, linewidth=2.5),
                 whiskerprops=dict(color=GREY), capprops=dict(color=GREY),
                 flierprops=dict(marker="o", markersize=3, markerfacecolor=GREY,
                                 markeredgecolor=GREY, alpha=0.5))
for box, q in zip(bp["boxes"], qs):
    box.set(facecolor=q_color[q], edgecolor=EDGE, linewidth=1)
save("3_boxplot_rainbow.png")

# ---------- 4) Heatmap: จำนวนออเดอร์ในแต่ละช่วง revenue x quantity (แดง = น้อย ... ม่วง = มาก) ----------
rev_edges = np.arange(0, df["revenue"].max() + 300, 300)
qty_edges = np.arange(min(qs) - 0.5, max(qs) + 1.5, 1)
counts, _, _ = np.histogram2d(df["quantity"], df["revenue"], bins=[qty_edges, rev_edges])

cmap_h = CMAP.copy()
cmap_h.set_bad("#FFFFFF")  # ช่องที่ไม่มีออเดอร์ให้เป็นสีขาว
ax = new_figure("Number of Orders by Quantity and Revenue Range", "Quantity", "Revenue Range", grid=False)
mesh = plt.pcolormesh(qty_edges, rev_edges, np.ma.masked_equal(counts.T, 0), cmap=cmap_h,
                      edgecolors="white", linewidth=0.8)
for i, q in enumerate(qs):
    for j in range(len(rev_edges) - 1):
        n = int(counts[i, j])
        if n > 0:
            plt.text(q, (rev_edges[j] + rev_edges[j + 1]) / 2, str(n), ha="center", va="center",
                     fontsize=8, color=text_color_on(cmap_h(n / counts.max())))
cb = plt.colorbar(mesh, pad=0.02)
cb.set_label("Number of Orders")
plt.xticks(qs)
ax.yaxis.set_major_formatter(comma)
save("4_heatmap_rainbow.png")

# ---------- 5) Donut: สัดส่วนรายได้ตาม quantity (สีรุ้งตาม quantity) ----------
share = total / total.sum() * 100
cols = [q_color[q] for q in total.index]

plt.figure(figsize=(8, 7), facecolor=BG)
plt.title("Revenue Share by Quantity", fontsize=14, fontweight="bold", color=TEXT, pad=15)
wedges, _ = plt.pie(total.values, colors=cols, startangle=90, counterclock=False,
                    wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2))
for w, pct, col in zip(wedges, share.values, cols):
    ang = np.deg2rad((w.theta1 + w.theta2) / 2)
    rr = 1 - 0.21  # กึ่งกลางวงแหวน
    plt.text(rr * np.cos(ang), rr * np.sin(ang), f"{pct:.1f}%", ha="center", va="center",
             fontsize=10, fontweight="bold", color=text_color_on(matplotlib.colors.to_rgb(col)))
plt.legend(wedges, [f"Quantity {q}" for q in total.index], loc="center left",
           bbox_to_anchor=(1.0, 0.5), frameon=False)
plt.text(0, 0, f"Total\n{total.sum():,.0f}", ha="center", va="center",
         fontsize=13, fontweight="bold", color=TEXT)
save("5_donut_rainbow.png")

saved C:\Users\User\Desktop\viewweiv\charts_rainbow\1_scatter_rainbow.png
saved C:\Users\User\Desktop\viewweiv\charts_rainbow\2_bar_rainbow.png
saved C:\Users\User\Desktop\viewweiv\charts_rainbow\3_boxplot_rainbow.png
saved C:\Users\User\Desktop\viewweiv\charts_rainbow\4_heatmap_rainbow.png
saved C:\Users\User\Desktop\viewweiv\charts_rainbow\5_donut_rainbow.png


In [22]:
import os
import sys

import matplotlib

matplotlib.use("Agg")  # ลบบรรทัดนี้ถ้าอยากให้แสดงหน้าต่างกราฟ แล้วเปลี่ยน savefig เป็น plt.show()
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

# ---------- ตำแหน่งไฟล์ ----------
CSV_PATH = r"C:\Users\User\Desktop\viewweiv\ecommerce_sales_analytics_5000.csv"

# กรอง argument ที่มาจาก Jupyter / IPykernel (ที่ขึ้นต้นด้วย '-') ออกไป
args = [a for a in sys.argv[1:] if not a.startswith("-")]

CSV = args[0] if len(args) > 0 else CSV_PATH
OUT_DIR = args[1] if len(args) > 1 else os.path.join(os.path.dirname(CSV), "charts_5")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- สี (ไล่สีม่วงจากอ่อนไปเข้ม) ----------
LIGHT = "#E6DFFA"        # ม่วงอ่อนสุด
DARK = "#4B3A9A"         # ม่วงเข้มสุด
EDGE = "#3B2D7A"         # สีขอบ
LINE = "#E05A67"         # เส้นตรง Least Squares (แดงชมพู)
GREY = "#A8A8A8"
TEXT = "#333333"
BG = "#FAFAFA"
cmap = LinearSegmentedColormap.from_list("light_to_dark_purple", [LIGHT, "#B19FF1", DARK])


def shade(n, lo=0.15, hi=1.0):
    """คืนลิสต์สี n สี ไล่จากอ่อนไปเข้ม"""
    return [cmap(v) for v in np.linspace(lo, hi, n)]


comma = matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}")


def new_figure(title, xlabel, ylabel, figsize=(10, 6), grid=True):
    plt.figure(figsize=figsize, facecolor=BG)
    ax = plt.axes()
    ax.set_facecolor("#FFFFFF")
    plt.title(title, fontsize=14, fontweight="bold", color=TEXT, pad=15)
    plt.xlabel(xlabel, fontsize=12, labelpad=10)
    plt.ylabel(ylabel, fontsize=12, labelpad=10)
    if grid:
        plt.grid(axis="y", linestyle=":", alpha=0.6, zorder=1)
    return ax


def save(name, tight_bbox=False):
    plt.tight_layout()
    path = os.path.join(OUT_DIR, name)
    kwargs = {"bbox_inches": "tight"} if tight_bbox else {}
    plt.savefig(path, dpi=150, facecolor=plt.gcf().get_facecolor(), **kwargs)
    plt.close()
    print("saved", path)


# ---------- โหลดข้อมูล ----------
df = pd.read_csv(CSV, encoding="utf-8-sig")[["quantity", "revenue"]]
qs = sorted(df["quantity"].unique())
qmin, qmax = qs[0], qs[-1]

# เส้นตรง Least Squares จากข้อมูลทั้งหมด
m, c = np.polyfit(df["quantity"], df["revenue"], 1)
r = df["quantity"].corr(df["revenue"])
x_line = np.linspace(qmin, qmax, 100)

# ============ 1) Linear (ปกติ): จุดข้อมูลดิบ + เส้นตรง + drop lines ============
sample = df.sample(n=40, random_state=42).sort_values("quantity").reset_index(drop=True)
xs, ys = sample["quantity"].values, sample["revenue"].values
fitted = m * xs + c

ax = new_figure("1) Linear Regression: Quantity vs Revenue", "Quantity", "Revenue")
ax.yaxis.set_major_formatter(comma)
plt.plot(x_line, m * x_line + c, color=LINE, linewidth=3,
         label=f"Least Squares Line (r = {r:.2f})", zorder=2)
for i in range(len(sample)):
    plt.vlines(xs[i], min(ys[i], fitted[i]), max(ys[i], fitted[i]),
               colors=GREY, linestyles="--", linewidth=1.2, zorder=3)
sc = plt.scatter(xs, ys, c=ys, cmap=cmap, s=60, edgecolors=EDGE, linewidths=1,
                 zorder=4, label="Data Points (random sample, n=40)")
plt.colorbar(sc, label="Revenue", format=comma, pad=0.02)
plt.xticks(qs)
plt.legend(frameon=True, facecolor="#FFFFFF", edgecolor="none", loc="upper left")
save("1_linear_regression.png")

# ============ 2) Linear (เฉลี่ย): ค่าเฉลี่ยของแต่ละ quantity + เส้นตรง ============
mean_rev = df.groupby("quantity")["revenue"].mean()

ax = new_figure("2) Linear Regression: Average Revenue by Quantity", "Quantity",
                "Average Revenue per Order")
ax.yaxis.set_major_formatter(comma)
plt.plot(x_line, m * x_line + c, color=LINE, linewidth=3,
         label="Least Squares Line (all orders)", zorder=2)
plt.scatter(mean_rev.index, mean_rev.values, c=shade(len(mean_rev)), s=130,
            edgecolors=EDGE, linewidths=1.2, zorder=4, label="Average per Quantity")
for q, v in mean_rev.items():
    plt.text(q, v + 70, f"{v:,.0f}", ha="center", fontsize=10, color=TEXT)
plt.ylim(0, mean_rev.max() * 1.18)
plt.xticks(qs)
plt.legend(frameon=True, facecolor="#FFFFFF", edgecolor="none", loc="upper left")
save("2_linear_average.png")

# ============ 3) Bar: รายได้รวมตาม quantity ============
total_rev = df.groupby("quantity")["revenue"].sum()

ax = new_figure("3) Total Revenue by Quantity", "Quantity", "Total Revenue")
ax.yaxis.set_major_formatter(comma)
bars = plt.bar(total_rev.index.astype(str), total_rev.values, color=shade(len(total_rev)),
               edgecolor=EDGE, linewidth=1, zorder=3)
for b, v in zip(bars, total_rev.values):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:,.0f}", ha="center",
             va="bottom", fontsize=10, color=TEXT)
plt.ylim(0, total_rev.max() * 1.1)
save("3_total_revenue_bar.png")

# ============ 4) Boxplot: การกระจาย revenue ตาม quantity (พร้อมบอกตัวเลขแต่ละค่า) ============
data = [df.loc[df["quantity"] == q, "revenue"] for q in qs]

ax = new_figure("4) Revenue Distribution by Quantity (Boxplot)", "Quantity", "Revenue per Order")
ax.yaxis.set_major_formatter(comma)

bp = plt.boxplot(data, labels=qs, patch_artist=True, widths=0.45,
                 medianprops=dict(color=LINE, linewidth=2.5),
                 whiskerprops=dict(color=GREY, linewidth=1.2),
                 capprops=dict(color=GREY, linewidth=1.2),
                 flierprops=dict(marker="o", markersize=3, markerfacecolor=GREY,
                                 markeredgecolor=GREY, alpha=0.5))

for box, col in zip(bp["boxes"], shade(len(qs))):
    box.set(facecolor=col, edgecolor=EDGE, linewidth=1)

# เพิ่มตัวเลขระบุค่าสถิติ (5-Number Summary) ประจำแต่ละ quantity
for idx, q in enumerate(qs, start=1):
    q_data = df.loc[df["quantity"] == q, "revenue"]
    q1 = q_data.quantile(0.25)
    med = q_data.median()
    q3 = q_data.quantile(0.75)
    iqr = q3 - q1
    
    whis_low = q_data[q_data >= q1 - 1.5 * iqr].min()
    whis_high = q_data[q_data <= q3 + 1.5 * iqr].max()

    offset_x = idx + 0.25  # ระยะเว้นข้อความขยับไปทางขวาของกล่อง
    
    # แสดงค่า Max, Q3, Median, Q1, Min
    ax.text(offset_x, whis_high, f"Max: {whis_high:,.0f}", va="center", fontsize=8, color="#555555")
    ax.text(offset_x, q3, f"Q3: {q3:,.0f}", va="center", fontsize=8, color=TEXT)
    ax.text(offset_x, med, f"Med: {med:,.0f}", va="center", fontsize=8, fontweight="bold", color=LINE)
    ax.text(offset_x, q1, f"Q1: {q1:,.0f}", va="center", fontsize=8, color=TEXT)
    ax.text(offset_x, whis_low, f"Min: {whis_low:,.0f}", va="center", fontsize=8, color="#555555")

plt.xlim(0.5, len(qs) + 0.8)
save("4_boxplot.png")

# ============ 5) Donut: สัดส่วนรายได้รวมของแต่ละ quantity ============
total = total_rev.sum()
share = total_rev / total * 100
colors = shade(len(total_rev))

fig, ax = plt.subplots(figsize=(9, 7), facecolor=BG)
wedges, _ = ax.pie(
    total_rev.values,
    colors=colors,
    startangle=90,
    counterclock=False,
    wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2),
)
for w, pct, col in zip(wedges, share.values, colors):
    ang = np.deg2rad((w.theta1 + w.theta2) / 2)
    rad = 1 - 0.42 / 2
    lum = 0.299 * col[0] + 0.587 * col[1] + 0.114 * col[2]
    ax.text(rad * np.cos(ang), rad * np.sin(ang), f"{pct:.1f}%", ha="center", va="center",
            fontsize=11, fontweight="bold", color="white" if lum < 0.55 else TEXT)

ax.text(0, 0.07, f"{total:,.0f}", ha="center", va="center", fontsize=22,
        fontweight="bold", color=TEXT)
ax.text(0, -0.1, "Total Revenue", ha="center", va="center", fontsize=12, color="#666666")
ax.set_title("5) Revenue Share by Quantity (Donut)", fontsize=14, fontweight="bold",
             color=TEXT, pad=20)
ax.legend(wedges, [f"Quantity {q}  ({v:,.0f})" for q, v in total_rev.items()],
          loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False, fontsize=11)

save("5_donut.png", tight_bbox=True)

saved C:\Users\User\Desktop\viewweiv\charts_5\1_linear_regression.png
saved C:\Users\User\Desktop\viewweiv\charts_5\2_linear_average.png
saved C:\Users\User\Desktop\viewweiv\charts_5\3_total_revenue_bar.png
saved C:\Users\User\Desktop\viewweiv\charts_5\4_boxplot.png


C:\Users\User\AppData\Local\Temp\ipykernel_12720\1172514016.py:128: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = plt.boxplot(data, labels=qs, patch_artist=True, widths=0.45,


saved C:\Users\User\Desktop\viewweiv\charts_5\5_donut.png
